# Initial modeling

This notebook orchestrates the modeling pipeline for the **VentureSurvive** project using the functions defined in `src/`.

In [11]:
# Imports principaux pour le pipeline de modélisation

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from src import data as data_mod
from src import features as features_mod
from src import split as split_mod
from src import model as model_mod
from src import evaluate as eval_mod


## Data loading and preparation

In this section, we load the raw data and apply the first cleaning steps: status filtering, label creation, date conversion, and funding amount cleaning.

In [12]:
# Chargement des données brutes
df_raw = data_mod.load_raw()
print("df_raw.shape:", df_raw.shape)

# Filtrage des statuts
df_filtered = data_mod.filter_status(df_raw)
print("df_filtered.shape (après filtrage des statuts):", df_filtered.shape)

# Création du label binaire success
df_labeled = data_mod.create_label(df_filtered)

# Conversion des dates
df_dates = data_mod.convert_dates(df_labeled)

# Nettoyage de la colonne funding_total_usd
df_clean = data_mod.clean_funding(df_dates)

print("df_clean.shape (après nettoyage):", df_clean.shape)

df_clean.head()

FileNotFoundError: Raw data file not found at data/startups_raw.csv

## Feature construction

In this section, we build time-based, geographic, and category features, then assemble a feature DataFrame ready for modeling.

In [ ]:
# Assemblage des features à partir du DataFrame nettoyé

df_features = features_mod.assemble_features(df_clean)

# Séparation des features X et de la cible y
y = df_features["success"].astype(int)
X = df_features.drop(columns=["success"])

print("df_features.shape:", df_features.shape)
print("Nombre de features (X):", X.shape[1])
print("Distribution de la cible success:\n", y.value_counts())

X.head()

## Train / test split

In this section, we perform a temporal train/test split based on `first_funding_at`, then optionally scale numeric features.

In [ ]:
# Ajout de X et y dans un même DataFrame pour le split temporel

df_for_split = df_features.copy()

# Split temporel en utilisant la fonction utilitaire
train_df, test_df = split_mod.temporal_split(df_for_split, cutoff_date="2013-01-01", date_col="first_funding_at")

print("train_df.shape:", train_df.shape)
print("test_df.shape:", test_df.shape)

# Séparation X / y pour train et test
y_train = train_df["success"].astype(int)
X_train = train_df.drop(columns=["success"])

y_test = test_df["success"].astype(int)
X_test = test_df.drop(columns=["success"])

# Optionnel : mise à l'échelle des features numériques
numeric_cols = X_train.select_dtypes(include=["number"]).columns
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

print("X_train_scaled.shape:", X_train_scaled.shape)
print("X_test_scaled.shape:", X_test_scaled.shape)

## Model training

In this section, we prepare the training of baseline models (Logistic Regression and Random Forest), but **they are not run until you manually execute the cells**.

In [ ]:
# Logistic Regression training (run when you are ready)

# logistic_model = model_mod.train_logistic(X_train_scaled.values, y_train.values)
# logistic_results = eval_mod.evaluate_classification(logistic_model, X_test_scaled.values, y_test.values)
# print("Logistic Regression results:", logistic_results)

# Random Forest training (run when you are ready)

# rf_model = model_mod.train_random_forest(X_train_scaled.values, y_train.values)
# rf_results = eval_mod.evaluate_classification(rf_model, X_test_scaled.values, y_test.values)
# print("Random Forest results:", rf_results)

## Évaluation et visualisations

Dans cette section, nous proposons des cellules pour tracer les courbes ROC / PR et la matrice de confusion, une fois les modèles entraînés.

In [ ]:
# Exemples d'utilisation des fonctions d'évaluation (à exécuter après entraînement des modèles)

# import matplotlib.pyplot as plt

# fig, axes = plt.subplots(1, 3, figsize=(18, 5))
# eval_mod.plot_roc(logistic_model, X_test_scaled.values, y_test.values, ax=axes[0])
# eval_mod.plot_pr(logistic_model, X_test_scaled.values, y_test.values, ax=axes[1])
# eval_mod.plot_confusion(logistic_model, X_test_scaled.values, y_test.values, normalize=True, ax=axes[2])
# plt.tight_layout()

# fig, axes = plt.subplots(1, 3, figsize=(18, 5))
# eval_mod.plot_roc(rf_model, X_test_scaled.values, y_test.values, ax=axes[0])
# eval_mod.plot_pr(rf_model, X_test_scaled.values, y_test.values, ax=axes[1])
# eval_mod.plot_confusion(rf_model, X_test_scaled.values, y_test.values, normalize=True, ax=axes[2])
# plt.tight_layout()